In [19]:
import os

print("\n📂 Files in /content:")
for f in os.listdir("/content"):
    print("➡", os.path.join("/content", f))



📂 Files in /content:
➡ /content/.config
➡ /content/cardihack_dummy_pred-1762193921064.csv
➡ /content/sample_data


In [20]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import log_loss, cohen_kappa_score
import warnings
warnings.filterwarnings("ignore")

# ===================================================
# CONFIG
# ===================================================
ID_COL = "trustii_id"
AGE_COL = "Age_Baseline"
SEX_COL = "Genre"
TARGET_SEVERITY = "OUTCOME SEVERITY"
TARGET_MACE = "OUTCOME MACE"

SNP_COLS_ALL = [f"SNP{i}" for i in range(1, 289)]
PRIORITY_SNP = [f"SNP{i}" for i in range(1, 76)]

# ===================================================
# LOAD DATA
# ===================================================
print("Loading data...")
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
dummy = pd.read_csv("/content/cardihack_dummy_pred-1762193921064.csv")

# ===================================================
# SNP FILTERING
# ===================================================
def filter_by_maf(df, snp_cols, maf_threshold=0.01):
    selected = []
    for snp in snp_cols:
        vals = df[snp].dropna()
        if len(vals) < 10: continue
        freq = vals.value_counts()
        if len(freq) < 2: continue
        maf = freq.min() / freq.sum()
        if maf >= maf_threshold: selected.append(snp)
    return selected

def chi2_select_snps(df, target, snp_cols, p_threshold=0.1):
    selected = []
    for snp in snp_cols:
        tab = pd.crosstab(df[snp], df[target])
        if tab.shape[0] < 2 or tab.shape[1] < 2: continue
        try:
            _, p, _, _ = chi2_contingency(tab)
            if p < p_threshold: selected.append(snp)
        except: pass
    return selected

def enforce_priority(train, selected, max_total=90):
    if not any(s in PRIORITY_SNP for s in selected):
        backup = chi2_select_snps(train, TARGET_MACE, PRIORITY_SNP, 0.2)
        if backup: selected = [backup[0]] + selected
    return selected[:max_total]

# ===================================================
# FEATURE SELECTION
# ===================================================
sev_snps = filter_by_maf(train, SNP_COLS_ALL, 0.01)
sev_snps = chi2_select_snps(train,TARGET_SEVERITY,sev_snps,0.1)
sev_features=[AGE_COL,SEX_COL]+sev_snps

clinical_cols=[c for c in train.columns if c not in SNP_COLS_ALL+[ID_COL,TARGET_MACE,TARGET_SEVERITY]]
mace_snps = filter_by_maf(train,SNP_COLS_ALL,0.01)
mace_snps = chi2_select_snps(train,TARGET_MACE,mace_snps,0.05)
mace_snps = enforce_priority(train,mace_snps,90)
mace_features=[AGE_COL,SEX_COL]+clinical_cols+mace_snps
mace_features=mace_features[:150]

# ===================================================
# MODELS (SCORING-AWARE)
# ===================================================
sev_model = Pipeline([
    ("imp",IterativeImputer(max_iter=15,random_state=42)),
    ("sc",StandardScaler()),
    ("clf",LogisticRegression(penalty="elasticnet",solver="saga",
                              l1_ratio=0.2,C=1.0,max_iter=6000))
])

mace_model = Pipeline([
    ("imp",IterativeImputer(max_iter=15,random_state=42)),
    ("sc",StandardScaler()),
    ("clf",LogisticRegression(penalty="elasticnet",solver="saga",
                              l1_ratio=0.2,C=1.0,multi_class="multinomial",max_iter=7000))
])

# ===================================================
# TRAIN
# ===================================================
print("Training models...")
sev_model.fit(train[sev_features],train[TARGET_SEVERITY])
mace_model.fit(train[mace_features],train[TARGET_MACE])

# ===================================================
# PREDICT FOR SUBMISSION
# ===================================================
severity_pred = sev_model.predict_proba(test[sev_features])[:,1]

mace_proba = mace_model.predict_proba(test[mace_features])
mace_pred = np.zeros(len(mace_proba),dtype=int)
mace_pred[mace_proba[:,2] > 0.45] = 2
mace_pred[(mace_proba[:,1] > 0.35) & (mace_proba[:,2] <= 0.45)] = 1

# ===================================================
# SCORE CALCULATION FUNCTION (NOW INTEGRATED)
# ===================================================
def compute_competition_score(train_df, sev_model, mace_model):
    sev_train_pred = sev_model.predict_proba(train_df[sev_features])[:,1]
    mace_train_pred = np.argmax(mace_model.predict_proba(train_df[mace_features]),axis=1)

    mace_w = compute_sample_weight({0:1,1:3,2:4},train_df[TARGET_MACE])
    sev_w  = compute_sample_weight({1:1,0:1.5},train_df[TARGET_SEVERITY])

    logloss = log_loss(train_df[TARGET_SEVERITY],sev_train_pred,sample_weight=sev_w)
    kappa = cohen_kappa_score(train_df[TARGET_MACE],mace_train_pred,
                              weights='quadratic',sample_weight=mace_w)

    random_ll=0.566
    rescaled = 1-((logloss-random_ll)/(1-random_ll))
    score = 0.7*kappa + 0.3*rescaled
    print(f"\n📊 Estimated Competition Score = {score:.4f}")
    return score

# Run score evaluation
compute_competition_score(train,sev_model,mace_model)

# ===================================================
# SUBMISSION MATCHING DUMMY FORMAT
# ===================================================
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_MACE: mace_pred,
    TARGET_SEVERITY: severity_pred
})

submission = submission[dummy.columns]     # column order EXACT match
submission.to_csv("submission_t.csv",index=False)
print("✅ submission_t.csv generated successfully")


Loading data...
Training models...

📊 Estimated Competition Score = 0.7087
✅ submission_t.csv generated successfully
